In [10]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

import torch

from nunchaku._C import ops

In [ ]:
DTYPE = torch.bfloat16
DEVICE = "cuda"

input = torch.randn(4096, 3072, device=DEVICE, dtype=DTYPE)
output = torch.zeros(4096, 3072 // 2, device=DEVICE, dtype=torch.uint8)
oscales = torch.zeros(3072 // 64, 4096, device=DEVICE, dtype=DTYPE)

with torch.no_grad():
    ops.quantize_w4a4_act(
        input, output, oscales, False, False
    )

In [25]:
o_rec = torch.repeat_interleave(output, 2, dim=1)
o_rec[:, ::2] = torch.bitwise_and(o_rec[:, ::2], 0x0F)
o_rec[:, 1::2] = torch.bitwise_right_shift(o_rec[:, 1::2], 4)
o_rec = o_rec.to(torch.int8)
o_rec = torch.where(o_rec > 7, o_rec - 16, o_rec)


#M = o_rec * torch.repeat_interleave(data["oscales"], 64, dim=0).T
((o_rec > 0) == (input > 0)).float()[:, 1::2].mean()

tensor(0.5000, device='cuda:0')

In [ ]:
kwargs["input"][:, ::2]

In [ ]:
o_rec[:, ::2]

In [ ]:
from nunchaku.ops.quantize import svdq_quantize_w4a4_act_fuse_lora_cuda

lora_down = torch.zeros_like(data["lora_down"])

svdq_quantize_w4a4_act_fuse_lora_cuda(M, lora_down=lora_down)

In [ ]:
lora_down

In [ ]:
data["smooth"]